# 🧠 Deepfake Detection — Training Notebook
**IIIT Bangalore Hackathon | Run on Colab T4 GPU**

### Steps:
1. Upload your dataset zip OR mount Google Drive
2. Run all cells top to bottom
3. Download `best_model.pth` and `predictions.csv` when done

> ⚡ **Runtime → Change runtime type → T4 GPU** before running!

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────────────
!pip install -q timm albumentations grad-cam

In [ ]:
# ── Cell 3: Mount Google Drive (put your dataset zip in Drive) ──────────
from google.colab import drive
drive.mount('/content/drive')

# ── Option A: Copy zip from Drive ────────────────────────────────────────
# Edit this path to match where you uploaded the zip in your Drive:
DRIVE_DATASET_ZIP = '/content/drive/MyDrive/hackathon_dataset.zip'

import os, shutil
if os.path.exists(DRIVE_DATASET_ZIP):
    print('Copying dataset zip from Drive...')
    shutil.copy(DRIVE_DATASET_ZIP, '/content/hackathon_dataset.zip')
    !unzip -q /content/hackathon_dataset.zip -d /content/
    print('Dataset extracted.')
else:
    print(f'ZIP not found at {DRIVE_DATASET_ZIP}.')
    print('Alternative: upload hackathon_dataset.zip using the Files panel on the left.')

In [ ]:
# ── Cell 3b: (Alternative) Upload zip directly if not using Drive ───────
# Uncomment this cell and comment out Cell 3 if uploading manually
# from google.colab import files
# uploaded = files.upload()   # Choose hackathon_dataset.zip
# !unzip -q hackathon_dataset.zip -d /content/

In [ ]:
# ── Cell 4: Find dataset root ────────────────────────────────────────────
import glob
# Find train_metadata.csv to locate dataset root
metas = glob.glob('/content/**/train_metadata.csv', recursive=True)
print('Found:', metas)
DATASET_ROOT = str(__import__('pathlib').Path(metas[0]).parent) if metas else '/content/hackathon_dataset'
print('Dataset root:', DATASET_ROOT)

In [ ]:
# ── Cell 5: Config ───────────────────────────────────────────────────────
import os
from pathlib import Path
import torch

# ── Paths ──
DATASET_ROOT   = Path(DATASET_ROOT)
TRAIN_REAL_DIR = DATASET_ROOT / 'train' / 'real'
TRAIN_FAKE_DIR = DATASET_ROOT / 'train' / 'fake'
TEST_DIR       = DATASET_ROOT / 'test'  / 'images'
TRAIN_META_CSV = DATASET_ROOT / 'train_metadata.csv'

os.makedirs('/content/weights',  exist_ok=True)
os.makedirs('/content/outputs',  exist_ok=True)
os.makedirs('/content/data',     exist_ok=True)

# ── Training settings ──
BACKBONE     = 'efficientnet_b4'
IMAGE_SIZE   = 224
BATCH_SIZE   = 32          # Increase to 64 if GPU has >8GB VRAM
EPOCHS       = 25
LR           = 1e-4
WEIGHT_DECAY = 1e-4
SEED         = 42
THRESHOLD    = 0.5
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()

print(f'Device : {DEVICE}')
print(f'AMP    : {USE_AMP}')
print(f'Train/ : {TRAIN_REAL_DIR.exists()}, Fake/ : {TRAIN_FAKE_DIR.exists()}, Test/ : {TEST_DIR.exists()}')

In [ ]:
# ── Cell 6: Reproducibility + Train/Val Split ────────────────────────────
import random, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(SEED)

meta = pd.read_csv(TRAIN_META_CSV)
train_df, val_df = train_test_split(meta, test_size=0.2, stratify=meta['label'], random_state=SEED)
train_df.to_csv('/content/data/train_split.csv', index=False)
val_df.to_csv('/content/data/val_split.csv',   index=False)

print(f'Train: {len(train_df)} | Val: {len(val_df)}')
print(f'Train fake/real: {(train_df.label==1).sum()} / {(train_df.label==0).sum()}')
print(f'Val   fake/real: {(val_df.label==1).sum()} / {(val_df.label==0).sum()}')

In [ ]:
# ── Cell 7: Dataset + Augmentations ─────────────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset, DataLoader

def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.4),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.3),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=7, p=1.0),
        ], p=0.3),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.ImageCompression(quality_lower=60, quality_upper=100, p=0.4),
        A.CoarseDropout(max_holes=4, max_height=32, max_width=32,
                        min_holes=1, min_height=8, min_width=8, p=0.2),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

class DeepfakeDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        df = pd.read_csv(csv_path)
        self.transform = transform
        self.image_paths, self.labels = [], []
        for _, row in df.iterrows():
            folder = TRAIN_FAKE_DIR if int(row['label']) == 1 else TRAIN_REAL_DIR
            path   = folder / f"{row['image_id']}.jpg"
            if path.exists():
                self.image_paths.append(path)
                self.labels.append(int(row['label']))
        print(f'[Dataset] {len(self.image_paths)} images (Real:{self.labels.count(0)} Fake:{self.labels.count(1)})')

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        img  = np.array(Image.open(self.image_paths[idx]).convert('RGB'))
        lbl  = self.labels[idx]
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(lbl, dtype=torch.float32)

train_ds = DeepfakeDataset('/content/data/train_split.csv', get_train_transforms())
val_ds   = DeepfakeDataset('/content/data/val_split.csv',   get_val_transforms())
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
print('DataLoaders ready.')

In [ ]:
# ── Cell 8: Model ────────────────────────────────────────────────────────
import torch.nn as nn
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights

def build_model():
    model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 1),
    )
    return model

model = build_model().to(DEVICE)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: EfficientNet-B4 | Trainable params: {params:,}')

In [ ]:
# ── Cell 9: Training Loop ────────────────────────────────────────────────
import time
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = GradScaler(enabled=USE_AMP)

best_auc, log_rows = 0.0, []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ── Train ──
    model.train()
    tr_loss, tr_preds, tr_labels = 0.0, [], []
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
        optimizer.zero_grad()
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        tr_loss += loss.item() * imgs.size(0)
        tr_preds.extend(torch.sigmoid(logits).detach().cpu().squeeze(1).tolist())
        tr_labels.extend(labels.cpu().squeeze(1).tolist())

    # ── Validate ──
    model.eval()
    vl_loss, vl_preds, vl_labels = 0.0, [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
            with autocast(enabled=USE_AMP):
                logits = model(imgs)
                loss   = criterion(logits, labels)
            vl_loss += loss.item() * imgs.size(0)
            vl_preds.extend(torch.sigmoid(logits).cpu().squeeze(1).tolist())
            vl_labels.extend(labels.cpu().squeeze(1).tolist())

    scheduler.step()

    tr_auc  = roc_auc_score(tr_labels, tr_preds)
    vl_auc  = roc_auc_score(vl_labels, vl_preds)
    vl_f1   = f1_score(vl_labels, (np.array(vl_preds) >= THRESHOLD).astype(int))
    elapsed = time.time() - t0

    print(f'Epoch [{epoch:02d}/{EPOCHS}] '
          f'TrLoss={tr_loss/len(train_ds):.4f} TrAUC={tr_auc:.4f} | '
          f'VlLoss={vl_loss/len(val_ds):.4f} VlAUC={vl_auc:.4f} '
          f'VlF1={vl_f1:.4f} | {elapsed:.1f}s')

    log_rows.append({'epoch': epoch, 'train_auc': tr_auc, 'val_auc': vl_auc,
                     'train_loss': tr_loss/len(train_ds), 'val_loss': vl_loss/len(val_ds),
                     'val_f1': vl_f1, 'lr': optimizer.param_groups[0]['lr']})

    if vl_auc > best_auc:
        best_auc = vl_auc
        torch.save(model.state_dict(), '/content/weights/best_model.pth')
        print(f'  ★ New best AUC: {best_auc:.4f} — saved.')

torch.save(model.state_dict(), '/content/weights/last_model.pth')
pd.DataFrame(log_rows).to_csv('/content/outputs/train_log.csv', index=False)
print(f'\n[DONE] Best Val AUC: {best_auc:.4f}')

In [ ]:
# ── Cell 10: Plot Training Curves ────────────────────────────────────────
import matplotlib.pyplot as plt

log = pd.read_csv('/content/outputs/train_log.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(log.epoch, log.train_loss, label='Train Loss')
axes[0].plot(log.epoch, log.val_loss,   label='Val Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(log.epoch, log.train_auc, label='Train AUC')
axes[1].plot(log.epoch, log.val_auc,   label='Val AUC')
axes[1].axhline(0.90, linestyle='--', color='green', alpha=0.5, label='0.90 target')
axes[1].set_title('AUC-ROC'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/outputs/training_curves.png', dpi=120)
plt.show()
print(f'Best Val AUC: {log.val_auc.max():.4f} at epoch {log.val_auc.idxmax()+1}')

In [ ]:
# ── Cell 11: Generate predictions.csv (official inference) ───────────────
from torchvision import transforms as T

# Load best model
model.load_state_dict(torch.load('/content/weights/best_model.pth', map_location=DEVICE))
model.eval()

infer_tf = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

test_paths = sorted(TEST_DIR.glob('*.jpg'))
print(f'Test images: {len(test_paths)}')

all_ids, all_probs = [], []
INFER_BATCH = 64

with torch.no_grad():
    for i in range(0, len(test_paths), INFER_BATCH):
        batch_paths = test_paths[i:i+INFER_BATCH]
        imgs = torch.stack([infer_tf(Image.open(p).convert('RGB')) for p in batch_paths]).to(DEVICE)
        if USE_AMP: imgs = imgs.half(); model.half()
        logits = model(imgs)
        probs  = torch.sigmoid(logits).float().cpu().squeeze(1).tolist()
        all_ids.extend([p.stem for p in batch_paths])
        all_probs.extend(probs)
        if (i // INFER_BATCH + 1) % 10 == 0:
            print(f'  Processed {i+INFER_BATCH}/{len(test_paths)}', end='\r')

predictions = pd.DataFrame({
    'image_id':   all_ids,
    'label':      (np.array(all_probs) >= THRESHOLD).astype(int).tolist(),
    'confidence': [round(p, 6) for p in all_probs],
})
predictions.to_csv('/content/predictions.csv', index=False)
print(f'\n[DONE] predictions.csv saved ({len(predictions)} rows)')
print(f'  Fake: {predictions.label.sum()} | Real: {(predictions.label==0).sum()}')
predictions.head()

In [ ]:
# ── Cell 12: Save all outputs to Google Drive ────────────────────────────
import shutil

DRIVE_OUT = '/content/drive/MyDrive/DeepfakeHackathon_Outputs'
os.makedirs(DRIVE_OUT, exist_ok=True)

# Copy key files
for src, dst in [
    ('/content/weights/best_model.pth',         f'{DRIVE_OUT}/best_model.pth'),
    ('/content/weights/last_model.pth',         f'{DRIVE_OUT}/last_model.pth'),
    ('/content/predictions.csv',                f'{DRIVE_OUT}/predictions.csv'),
    ('/content/outputs/train_log.csv',           f'{DRIVE_OUT}/train_log.csv'),
    ('/content/outputs/training_curves.png',     f'{DRIVE_OUT}/training_curves.png'),
]:
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'  Saved → {dst}')

print('\n✅ All outputs saved to Google Drive!')
print(f'   Drive folder: {DRIVE_OUT}')

In [ ]:
# ── Cell 13: (Optional) Direct download if not using Drive ───────────────
# from google.colab import files
# files.download('/content/weights/best_model.pth')
# files.download('/content/predictions.csv')
# files.download('/content/outputs/train_log.csv')